Data Registration

In [10]:
from huggingface_hub import notebook_login
notebook_login()

In [11]:
from huggingface_hub import HfApi

api = HfApi()

api.create_repo(repo_id="Roshini-Boddepalli/tourism-dataset", repo_type="dataset", exist_ok=True)

api.upload_file(
    path_or_fileobj="tourism.csv",
    path_in_repo="tourism.csv",
    repo_id="Roshini-Boddepalli/tourism-dataset",
    repo_type="dataset"
)

CommitInfo(commit_url='https://huggingface.co/datasets/Roshini-Boddepalli/tourism-dataset/commit/f17e41a780b659694032ac98e006f734e1470812', commit_message='Upload tourism.csv with huggingface_hub', commit_description='', oid='f17e41a780b659694032ac98e006f734e1470812', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Roshini-Boddepalli/tourism-dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Roshini-Boddepalli/tourism-dataset'), pr_revision=None, pr_num=None)

The dataset was successfully uploaded to Hugging Face using an access token.
A dataset repository named 'tourism-dataset' was created, and the dataset is now
available for further stages of the MLOps pipeline.

In [14]:
# Data Preparation

# Remove extra column

df.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200005,0,32.0,Company Invited,1,8.0,Salaried,Male,3,3.0,Basic,3.0,Single,1.0,0,5,1,1.0,Executive,18068.0


In [15]:
# Checking missing values

df.isnull().sum()

,0
CustomerID,0
ProdTaken,0
Age,0
TypeofContact,0
CityTier,0
DurationOfPitch,0
Occupation,0
Gender,0
NumberOfPersonVisiting,0
NumberOfFollowups,0


In [20]:
# Handle missing values
df.ffill(inplace=True)

In [18]:
# Encode categorical data

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

In [19]:
# Split Data

from sklearn.model_selection import train_test_split

X = df.drop("ProdTaken", axis=1)
y = df["ProdTaken"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Observation : Data preprocessing was performed by handling missing values, removing unnecessary columns, encoding categorical variables, and splitting the dataset into training and testing sets.

In [21]:
# Model Building with Experimentation Tracking
# Load data from hugging face

from datasets import load_dataset
import pandas as pd

dataset = load_dataset("Roshini-Boddepalli/tourism-dataset")

df = dataset['train'].to_pandas()
df.head()

tourism.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/4128 [00:00<?, ? examples/s]

,Unnamed: 0,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,...,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,...,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,...,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,...,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,...,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,5,200005,0,32.0,Company Invited,1,8.0,Salaried,Male,3,...,Basic,3.0,Single,1.0,0,5,1,1.0,Executive,18068.0


Observation : The dataset was successfully loaded from the Hugging Face data repository for model building.

In [22]:
# Data Prep

df.ffill(inplace=True)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

from sklearn.model_selection import train_test_split

X = df.drop("ProdTaken", axis=1)
y = df["ProdTaken"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [23]:
#  Define a model and parameters

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()

params = {
    "n_estimators": [50, 100],
    "max_depth": [5, 10]
}

Observation : A Random Forest model was selected, and hyperparameters such as number of estimators and maximum depth were defined for tuning.

In [26]:
# Hyperparameter Tuning

from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(model, params, cv=3)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

Best Parameters: {'max_depth': 10, 'n_estimators': 100}
Best Score: 0.8706820246057303


Observation : Hyperparameter tuning was performed using Grid Search CV. The best combination of parameters was identified as max_depth = 10 and n_estimators = 100, which resulted in the highest cross-validation score of 0.87. This ensures improved model performance.

In [27]:
# Log Parameters

best_params = grid.best_params_
print("Best Parameters:", best_params)

Best Parameters: {'max_depth': 10, 'n_estimators': 100}


Observation : The best hyperparameters obtained after tuning were recorded for experiment tracking and reproducibility.

In [28]:
# Evaluation Model

from sklearn.metrics import accuracy_score

y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.887409200968523


Observation : The model was evaluated using accuracy score, which indicates how well the model predicts the target variable.

In [29]:
# Register model to hugging face

from huggingface_hub import HfApi

api = HfApi()

api.create_repo(repo_id="Roshini-Boddepalli/tourism-model", repo_type="model", exist_ok=True)

import joblib
joblib.dump(best_model, "model.pkl")

api.upload_file(
    path_or_fileobj="model.pkl",
    path_in_repo="model.pkl",
    repo_id="Roshini-Boddepalli/tourism-model",
    repo_type="model"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  model.pkl                   :  81%|########  | 2.42MB / 2.99MB            

CommitInfo(commit_url='https://huggingface.co/Roshini-Boddepalli/tourism-model/commit/e5690a3c2efd0bec185191d9aa20800d201ea202', commit_message='Upload model.pkl with huggingface_hub', commit_description='', oid='e5690a3c2efd0bec185191d9aa20800d201ea202', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Roshini-Boddepalli/tourism-model', endpoint='https://huggingface.co', repo_type='model', repo_id='Roshini-Boddepalli/tourism-model'), pr_revision=None, pr_num=None)

Observation : The trained Random Forest model was successfully saved and uploaded to the Hugging Face Model Hub. This allows the model to be easily accessed, shared, and reused for future predictions and deployment.


In [30]:
from google.colab import files
files.download("model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model Deployment :  

Observation: The trained Random Forest model was successfully saved and uploaded to the Hugging Face Model Hub. This allows the model to be easily accessed, shared, and reused for future predictions and deployment.

Model Loading:

The trained model was loaded from the Hugging Face Model Hub using the huggingface_hub library, enabling seamless integration and reuse of the model in the deployed application.

Dockerfile :

A Dockerfile was defined to specify the runtime environment for the application. It includes Python installation, dependency setup using requirements.txt, and execution of the application script (app.py).

Dependecies File :

A requirements.txt file was created to list all necessary Python libraries such as pandas, scikit-learn, joblib, and gradio required for running the application.

Input Processing:

User inputs collected through the Gradio interface were converted into a pandas DataFrame format, which is required for making predictions using the trained machine learning model.

Hosting Script :

The app.py file acts as the hosting script. It loads the trained model, accepts user inputs through a Gradio interface, processes the inputs into a dataframe, and returns predictions.